# SSAS Trace Timing with Semantic Link Labs

This notebook provides a Python function that mirrors the PowerShell trace pattern: start trace, run DAX, correlate trace events, and compute total/SE/FE timings.

In [ ]:
%pip install semantic-link-labs

In [ ]:
import time
import uuid
from typing import Optional

import pandas as pd
import sempy.fabric as fabric
import sempy_labs as labs

In [ ]:
def capture_ssas_trace_timings(
    dataset: str,
    dax_query: str,
    workspace: Optional[str] = None,
    clear_cache: bool = True,
    post_wait_seconds: float = 1.2
) -> pd.DataFrame:
    """
    Runs a DAX query with an SSAS trace and returns a one-row summary DataFrame.

    Parameters
    ----------
    dataset : str
        Semantic model name or ID.
    dax_query : str
        DAX query to execute.
    workspace : str, default=None
        Workspace name or ID.
    clear_cache : bool, default=True
        If True, calls sempy_labs.clear_cache before query execution.
    post_wait_seconds : float, default=1.2
        Wait time after query execution before stopping trace.

    Returns
    -------
    pandas.DataFrame
        Single-row summary with total, storage engine, and formula engine timings.
    """

    event_schema = {
        "QueryBegin": ["EventClass", "CurrentTime", "SessionID", "ActivityID", "RequestID", "TextData"],
        "QueryEnd": ["EventClass", "CurrentTime", "Duration", "CpuTime", "SessionID", "ActivityID", "RequestID", "TextData"],
        "VertiPaqSEQueryBegin": ["EventClass", "CurrentTime", "SessionID", "ActivityID", "RequestID", "TextData"],
        "VertiPaqSEQueryEnd": ["EventClass", "CurrentTime", "Duration", "CpuTime", "SessionID", "ActivityID", "RequestID", "TextData"],
        "VertiPaqSEQueryCacheMatch": ["EventClass", "CurrentTime", "SessionID", "ActivityID", "RequestID", "TextData"],
        "DirectQueryBegin": ["EventClass", "CurrentTime", "SessionID", "ActivityID", "RequestID", "TextData"],
        "DirectQueryEnd": ["EventClass", "CurrentTime", "Duration", "CpuTime", "SessionID", "ActivityID", "RequestID", "TextData"]
    }

    if clear_cache:
        labs.clear_cache(dataset=dataset, workspace=workspace)

    run_id = uuid.uuid4().hex
    run_marker = f"-- RUNID:{run_id}"
    tagged_query = f"{run_marker}\n{dax_query}"

    t0 = time.perf_counter()
    with fabric.create_trace_connection(dataset=dataset, workspace=workspace) as trace_connection:
        with trace_connection.create_trace(event_schema) as trace:
            trace.start()
            _ = fabric.evaluate_dax(dataset=dataset, workspace=workspace, dax_string=tagged_query)
            time.sleep(post_wait_seconds)
            df = trace.stop()
    t1 = time.perf_counter()

    if df is None or df.empty:
        elapsed = round((t1 - t0) * 1000.0, 3)
        summary = {
            "dataset": dataset,
            "workspace": workspace,
            "total_elapsed_ms": elapsed,
            "storage_engine_ms": 0.0,
            "formula_engine_ms": elapsed,
            "trace_event_count": 0,
            "note": "No trace events returned."
        }
        return pd.DataFrame([summary])

    event_class_col = None
    for c in ["Event Class", "EventClass"]:
        if c in df.columns:
            event_class_col = c
            break

    text_col = None
    for c in ["Text Data", "TextData"]:
        if c in df.columns:
            text_col = c
            break

    duration_col = "Duration" if "Duration" in df.columns else None

    current_time_col = None
    for c in ["Current Time", "CurrentTime"]:
        if c in df.columns:
            current_time_col = c
            break

    session_col = None
    for c in ["Session ID", "SessionID"]:
        if c in df.columns:
            session_col = c
            break

    activity_col = None
    for c in ["Activity ID", "ActivityID"]:
        if c in df.columns:
            activity_col = c
            break

    request_col = None
    for c in ["Request ID", "RequestID"]:
        if c in df.columns:
            request_col = c
            break

    if event_class_col is None:
        raise ValueError(f"Trace output does not contain event class column. Columns: {list(df.columns)}")

    if current_time_col is not None:
        df[current_time_col] = pd.to_datetime(df[current_time_col], errors="coerce")

    query_end_mask = df[event_class_col].astype(str).eq("QueryEnd")
    if text_col is not None:
        query_end_mask = query_end_mask & df[text_col].astype(str).str.contains(run_marker, regex=False, na=False)

    target = df.loc[query_end_mask].tail(1)
    if target.empty:
        target = df.loc[df[event_class_col].astype(str).eq("QueryEnd")].tail(1)

    corr = pd.Series(True, index=df.index)
    if not target.empty:
        target_row = target.iloc[0]

        target_session = str(target_row[session_col]) if session_col and pd.notna(target_row[session_col]) else ""
        target_request = str(target_row[request_col]) if request_col and pd.notna(target_row[request_col]) else ""
        target_activity = str(target_row[activity_col]) if activity_col and pd.notna(target_row[activity_col]) else ""

        if target_request and request_col is not None:
            corr = corr & (df[request_col].astype(str).eq(target_request))
            if session_col is not None and target_session:
                corr = corr & (df[session_col].astype(str).eq(target_session))
        elif target_activity and activity_col is not None:
            corr = corr & (df[activity_col].astype(str).eq(target_activity))
            if session_col is not None and target_session:
                corr = corr & (df[session_col].astype(str).eq(target_session))
        elif target_session and session_col is not None:
            corr = corr & (df[session_col].astype(str).eq(target_session))

    trace_df = df.loc[corr].copy()

    if duration_col is not None:
        trace_df[duration_col] = pd.to_numeric(trace_df[duration_col], errors="coerce")

    se_end_classes = {"VertiPaqSEQueryEnd", "DirectQueryEnd"}
    se_df = trace_df.loc[trace_df[event_class_col].astype(str).isin(se_end_classes)].copy()

    intervals: list[tuple[pd.Timestamp, pd.Timestamp]] = []
    if current_time_col is not None and duration_col is not None and not se_df.empty:
        for _, row in se_df.iterrows():
            end_ts = row[current_time_col]
            dur = row[duration_col]
            if pd.isna(end_ts):
                continue
            if pd.notna(dur) and float(dur) > 0:
                start_ts = end_ts - pd.to_timedelta(float(dur), unit="ms")
            else:
                start_ts = end_ts
            if end_ts >= start_ts:
                intervals.append((start_ts, end_ts))

    se_ms = 0.0
    if intervals:
        ordered = sorted(intervals, key=lambda x: (x[0], x[1]))
        se_total_ms = 0.0
        cur_start, cur_end = ordered[0]
        for start, end in ordered[1:]:
            if start <= cur_end:
                if end > cur_end:
                    cur_end = end
            else:
                se_total_ms += (cur_end - cur_start).total_seconds() * 1000.0
                cur_start, cur_end = start, end
        se_total_ms += (cur_end - cur_start).total_seconds() * 1000.0
        se_ms = round(se_total_ms, 3)

    total_ms = round((t1 - t0) * 1000.0, 3)
    if not target.empty and duration_col is not None:
        qe_duration = pd.to_numeric(target[duration_col], errors="coerce").iloc[-1]
        if pd.notna(qe_duration) and float(qe_duration) > 0:
            total_ms = round(float(qe_duration), 3)

    fe_ms = round(max(0.0, total_ms - se_ms), 3)

    summary = {
        "dataset": dataset,
        "workspace": workspace,
        "total_elapsed_ms": total_ms,
        "storage_engine_ms": se_ms,
        "formula_engine_ms": fe_ms,
        "trace_event_count": int(len(trace_df)),
        "note": "Timings are derived from QueryEnd + SE end events (VertiPaq/DirectQuery)."
    }

    return pd.DataFrame([summary])

## Run Example

Set your dataset/workspace and query, then execute the cell.

In [ ]:
dataset_name = ""  # e.g. 'Sales Model'
workspace_name = None  # e.g. 'My Workspace'
query_text = 'EVALUATE ROW("Sales", [Total Sales])'

summary_df = capture_ssas_trace_timings(
    dataset=dataset_name,
    workspace=workspace_name,
    dax_query=query_text,
    clear_cache=True
)

summary_df